# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mathu2112/FlyRank-ML-Internship-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## One row represents the daily performance metrics of one content item for one client on a single reporting date.THe analysis is restricted to the March 2026 as analysis window because it is a mid-panel month and avoids using the final month as an outcome/test period.

In [6]:
# This cell is for CODE (numbers, a query, a check).

!pip install -q datasets duckdb pyarrow pandas huggingface_hub

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

#For Accessing dataset
from datasets import load_dataset

dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)
print(dataset)

train = dataset["train"]
print(train.features["report_date"])
print(train[0]["report_date"])
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})
Value('date32')
2025-01-27


In [7]:

#for verifying for duplicates if any
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns="fact_content_daily_performance/month=2026-03/*"
)

print(dataset_path)

import os
import duckdb

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        print(os.path.join(root, file))

con = duckdb.connect()

result = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_keys
    FROM read_parquet("/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet")
""").fetchone()

print("Total rows:", result[0])
print("Unique keys:", result[1])
print("Duplicate rows:", result[0] - result[1])

#for verifying slice's count and date span

date_check = con.execute(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{"/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"}')
""").fetchone()

print("Row count:", date_check[0])
print("First date:", date_check[1])
print("Last date:", date_check[2])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/README.md
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-02/data_0.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-05/data_0.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2025-11/data_0.parquet
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2025-03/data_0.parquet
/root/.cache/huggingface/hub/d

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 9841378
Unique keys: 9841378
Duplicate rows: 0
Row count: 9841378
First date: 2026-03-01
Last date: 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


---
 For the assignment I chose the content performance / organic search lane, because the table has very useful Search Console variables like impressions,clicks,average position,GSC availability.

ane: Content performance / organic search

Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, and scroll_events. These are observed performance and engagement measurements from the completed reporting window.

Label / proxy: A future organic-search performance outcome derived from a later reporting window. The label will represent the outcome we are trying to predict/rank rather than a measurement from the same feature window.

Context: client_hash_id, content_hash_id, and report_date are used to identify and group observations and provide the time context.

Excluded: I deliberately exclude June 2026 from development because it is the final month and should be treated as a sealed outcome/test period. I also exclude identifiers from the feature set because they are grouping keys rather than meaningful performance measurements.


In [8]:
# This cell is for CODE (numbers, a query, a check).

train = dataset["train"]
print(train[:5])
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


{'report_date': [datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27), datetime.date(2025, 1, 27)], 'client_hash_id': ['client_9958f0a7ae1df715', 'client_9958f0a7ae1df715', 'client_9958f0a7ae1df715', 'client_9958f0a7ae1df715', 'client_9958f0a7ae1df715'], 'content_hash_id': ['content_3b70a18ea133b2bb', 'content_fe8e8155ce1d47a2', 'content_b4462a1b90640058', 'content_c899aef92518c714', 'content_c7c1d2e68d9d0964'], 'client_has_gsc': [True, True, True, True, True], 'client_has_ga4': [True, True, True, True, True], 'gsc_data_available': [True, True, True, True, True], 'ga4_data_available': [False, False, False, False, False], 'gsc_impressions': [30, 5, 1, 6, 5], 'gsc_clicks': [0, 0, 0, 0, 0], 'gsc_sum_position': [115, 358, 34, 140, 89], 'gsc_avg_position': [3.8333333333333335, 71.6, 34.0, 23.333333333333332, 17.8], 'ga4_pageviews': [0, 0, 0, 0, 0], 'ga4_sessions': [0, 0, 0, 0, 0], 'ga4_users': [0, 0, 0, 0, 0], 'ga4_engaged_sessions':

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


## Observation of Output:-

The March 2026 slice contains 9,841,378 rows from 2026-03-01 to 2026-03-31. The combination of report_date,client_hash_id, and content_hash_id is unique for all rows in this slice, with 0 duplicate keys. Of these rows, 3,611,061 have GSC data available when filtering with gsc_data_available IS TRUE.


I use five observable features from the completed March 2026 reporting window: GSC impressions,GSC clicks,GSC average position,GA4 sessions,and scroll events.These are measurements available from the completed reporting period before a subsequent outcome window is evaluated. I excluded identifiers such as client and content hashes from the feature set because they are grouping keys rather than meaningful measurements.

| Feature            | Available when?                                                                                                                  |
| ------------------ | -------------------------------------------------------------------------------------------------------------------------------- |
| `gsc_impressions`  | Knowable after the completed reporting period because the March Search Console impressions have already been recorded.           |
| `gsc_clicks`       | Knowable after the completed reporting period because the March Search Console clicks have already been recorded.                |
| `gsc_avg_position` | Knowable after the completed reporting period because the March Search Console position measurements have already been recorded. |
| `ga4_sessions`     | Knowable after the completed reporting period when GA4 data is available for the observation.                                    |
| `scroll_events`    | Knowable after the completed reporting period when the engagement data is available.                                             |



In [20]:
# This cell is for CODE (numbers, a query, a check).

# ============================================================
# WEEK 03 - DATA CONTRACT
# Verification queries + five features + leakage experiment
# ============================================================

import pandas as pd
import duckdb
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# ------------------------------------------------------------
# 1. Connect to DuckDB
# ------------------------------------------------------------

con = duckdb.connect()

# March 2026 parquet file
march_path = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"

# April 2026 parquet file
april_path = "/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-04/data_0.parquet"


# ============================================================
# 2. VERIFICATION QUERY 1 - GRAIN
# ============================================================

grain_check = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_keys
FROM read_parquet('{"/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"}')
""").fetchone()

print("Verification Query 1 - Grain")
print("Total rows:", grain_check[0])
print("Unique keys:", grain_check[1])
print("Duplicate rows:", grain_check[0] - grain_check[1])


# ============================================================
# 3. VERIFICATION QUERY 2 - ROW COUNT AND DATE SPAN
# ============================================================

date_check = con.execute(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{"/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"}')
""").fetchone()

print("\nVerification Query 2 - Row count and date span")
print("Row count:", date_check[0])
print("First date:", date_check[1])
print("Last date:", date_check[2])


# ============================================================
# 4. VERIFICATION QUERY 3 - GSC AVAILABILITY
# ============================================================

availability_check = con.execute(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{"/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"}')
WHERE gsc_data_available IS TRUE
""").fetchone()

print("\nVerification Query 3 - GSC availability")
print("Rows with GSC data available:", availability_check[0])


# ============================================================
# 5. FIVE-FEATURE FRAME
# ============================================================

features = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    scroll_events
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
""").df()

print("\nFive-feature frame")
print("Shape:", features.shape)

display(features.head())


# ============================================================
# 6. FUTURE OUTCOME - APRIL 2026
# ============================================================

# Future label:
# 1 = content received at least one GSC click during April
# 0 = content received no GSC clicks during April

future_label = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    MAX(
        CASE
            WHEN gsc_clicks > 0 THEN 1
            ELSE 0
        END
    ) AS future_clicked
FROM read_parquet('{april_path}')
WHERE gsc_data_available IS TRUE
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

print("\nFuture April outcome")
print(future_label["future_clicked"].value_counts())


# ============================================================
# 7. JOIN MARCH FEATURES WITH APRIL OUTCOME
# ============================================================

model_frame = features.merge(
    future_label,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("\nModel frame shape:", model_frame.shape)

display(model_frame.head())


# ============================================================
# 8. DELIBERATE LABEL-LEAKAGE EXPERIMENT
# ============================================================

# PURPOSE:
# Deliberately copy the future label into a feature.
# This is NOT an honest feature.
# It is done only to demonstrate leakage.

model_frame["LEAKED_LABEL_COPY"] = model_frame["future_clicked"]


# Use ONLY the deliberately leaked feature

X = model_frame[["LEAKED_LABEL_COPY"]]
y = model_frame["future_clicked"]


# Split the data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Train a simple model

leak_model = DecisionTreeClassifier(random_state=42)

leak_model.fit(X_train, y_train)


# Predict

leaked_pred = leak_model.predict(X_test)


# Calculate score

leaked_accuracy = accuracy_score(y_test, leaked_pred)

print("\nDeliberate leakage experiment")
print("Accuracy with deliberate label leakage:", leaked_accuracy)


# ============================================================
# 9. REMOVE THE LEAKED FEATURE
# ============================================================

model_frame = model_frame.drop(
    columns=["LEAKED_LABEL_COPY"]
)

print("\nLeaked feature removed.")

print("Remaining columns:")
print(model_frame.columns.tolist())
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification Query 1 - Grain
Total rows: 9841378
Unique keys: 9841378
Duplicate rows: 0

Verification Query 2 - Row count and date span
Row count: 9841378
First date: 2026-03-01
Last date: 2026-03-31

Verification Query 3 - GSC availability
Rows with GSC data available: 3611061


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Five-feature frame
Shape: (3611061, 8)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>



Future April outcome
future_clicked
0    126928
1     67832
Name: count, dtype: int64

Model frame shape: (3536466, 9)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events,future_clicked
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,<NA>,<NA>,1
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0.000000,<NA>,<NA>,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,<NA>,<NA>,1
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,4.000000,<NA>,<NA>,0
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,<NA>,<NA>,1



Deliberate leakage experiment
Accuracy with deliberate label leakage: 1.0

Leaked feature removed.
Remaining columns:
['client_hash_id', 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events', 'future_clicked']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

---

Limitation: This March 2026 slice represents only one month of warehouse observations, so it may not capture seasonal or longer-term changes in content performance. In addition, GSC and GA4 availability is not uniform across observations, so some measurements may be missing even when the content-client-date row exists. The slice therefore supports observed, directional analysis but cannot by itself establish long-term causal effects or guarantee performance outside the observed period

In [16]:
# This cell is for CODE (numbers, a query, a check).

availability_check = con.execute(f"""
    SELECT
        COUNT(*) AS available_rows
    FROM read_parquet('{"/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet"}')
    WHERE gsc_data_available IS TRUE
""").fetchone()

print("Rows with GSC data available:", availability_check[0])
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Rows with GSC data available: 3611061


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.